# LakeSignal — Daily Backtest
Compare yesterday's AI predictions against actual stock price movements.  
Runs daily after market close to score prediction accuracy.

In [0]:
%pip install yfinance>=0.2.0 openai>=1.60.0 feedparser==6.0.11
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("catalog", "lakesignal")
dbutils.widgets.text("schema", "core")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
T_IMPACT = f"{CATALOG}.{SCHEMA}.impact_analysis"
T_NEWS = f"{CATALOG}.{SCHEMA}.news_events"
T_BACKTEST = f"{CATALOG}.{SCHEMA}.backtest_results"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")
print(f"Backtest target: {T_BACKTEST}")

## Step 1: Find impacts needing backtest scores
Get all impacts that either have no backtest row, or where actual prices were previously unavailable.

In [0]:
unscored = spark.sql(f"""
    SELECT ia.impact_id, ia.event_id, ia.ticker_symbol as ticker, ia.direction,
           ia.magnitude, ia.predicted_move_pct_1d, ia.predicted_move_pct_5d,
           ia.confidence, ia.analyzed_at,
           DATE(ia.analyzed_at) as event_date,
           ne.headline,
           bt.backtest_id as existing_bt_id
    FROM {T_IMPACT} ia
    JOIN {T_NEWS} ne ON ne.event_id = ia.event_id
    LEFT JOIN {T_BACKTEST} bt ON bt.impact_id = ia.impact_id
    WHERE bt.impact_id IS NULL
       OR bt.actual_close_t1 IS NULL
       OR bt.actual_close_t5 IS NULL
    ORDER BY ia.analyzed_at
""").toPandas()

print(f"Impacts to score: {len(unscored)}")
print(f"Unique tickers: {unscored['ticker'].nunique()}")
if len(unscored) > 0:
    print(f"Date range: {unscored['event_date'].min()} to {unscored['event_date'].max()}")

## Step 2: Fetch actual stock prices from Yahoo Finance

In [0]:
import yfinance as yf
import pandas as pd
from datetime import timedelta

if len(unscored) == 0:
    print("No impacts to backtest. Exiting.")
    dbutils.notebook.exit("no_impacts")

tickers = sorted(unscored['ticker'].unique().tolist())
min_date = pd.Timestamp(unscored['event_date'].min()) - timedelta(days=3)
max_date = pd.Timestamp(unscored['event_date'].max()) + timedelta(days=12)

print(f"Fetching prices for {len(tickers)} tickers from {min_date.date()} to {max_date.date()}...")
prices_raw = yf.download(tickers, start=str(min_date.date()), end=str(max_date.date()), progress=False)

# Build clean close-price lookup: {ticker: {date: close_price}}
# yfinance returns MultiIndex: ('Close', 'AAPL') — access via prices_raw['Close'][ticker]
close_lookup = {}
if len(tickers) == 1:
    close_lookup[tickers[0]] = {d.date(): float(v) for d, v in prices_raw['Close'].dropna().items()}
else:
    close_df = prices_raw['Close']
    for t in tickers:
        try:
            series = close_df[t].dropna()
            close_lookup[t] = {d.date(): float(v) for d, v in series.items()}
        except Exception as e:
            print(f"  Warning: no data for {t}: {e}")

print(f"Got price data for {len(close_lookup)}/{len(tickers)} tickers")
all_dates = set()
for t in close_lookup:
    all_dates.update(close_lookup[t].keys())
print(f"Trading days available: {sorted(all_dates)}")

## Step 3: Compare predictions vs actual prices
For each impact:
- **t0** = closing price on the last trading day BEFORE the prediction (baseline)
- **t+1** = closing price on the prediction day (1-day impact)
- **t+5** = closing price 5 trading days after t0 (5-day impact)

In [0]:
import uuid
from datetime import datetime, timezone, date as date_type
import numpy as np

def get_trading_days(lookup, after_date):
    return sorted([d for d in lookup.keys() if d > after_date])

results = []
now = datetime.now(timezone.utc)

for _, row in unscored.iterrows():
    ticker = row['ticker']
    event_dt = row['event_date']
    if hasattr(event_dt, 'date') and callable(event_dt.date):
        event_dt = event_dt.date()
    elif isinstance(event_dt, str):
        event_dt = date_type.fromisoformat(str(event_dt))

    lookup = close_lookup.get(ticker, {})
    if not lookup:
        continue

    sorted_dates = sorted(lookup.keys())

    # t0 = last trading day close BEFORE event date
    t0_candidates = [d for d in sorted_dates if d < event_dt]
    t0_date = t0_candidates[-1] if t0_candidates else None
    t0_close = lookup.get(t0_date) if t0_date else None

    # t1 = event day close (or first trading day on/after event date)
    t1_candidates = [d for d in sorted_dates if d >= event_dt]
    t1_date = t1_candidates[0] if t1_candidates else None
    t1_close = lookup.get(t1_date) if t1_date else None

    # t5 = 5th trading day after t0
    if t0_date:
        future_days = get_trading_days(lookup, t0_date)
        t5_date = future_days[4] if len(future_days) >= 5 else None
        t5_close = lookup.get(t5_date) if t5_date else None
    else:
        t5_date, t5_close = None, None

    # Calculate actual moves
    actual_1d = round((t1_close - t0_close) / t0_close * 100, 4) if (t0_close and t1_close) else None
    actual_5d = round((t5_close - t0_close) / t0_close * 100, 4) if (t0_close and t5_close) else None

    # Direction accuracy
    dir_correct_1d = None
    if actual_1d is not None:
        d = row['direction']
        if d == 'positive': dir_correct_1d = actual_1d > 0
        elif d == 'negative': dir_correct_1d = actual_1d < 0
        else: dir_correct_1d = abs(actual_1d) < 1.0

    dir_correct_5d = None
    if actual_5d is not None:
        d = row['direction']
        if d == 'positive': dir_correct_5d = actual_5d > 0
        elif d == 'negative': dir_correct_5d = actual_5d < 0
        else: dir_correct_5d = abs(actual_5d) < 2.0

    mag_err_1d = round(abs(float(row['predicted_move_pct_1d']) - actual_1d), 4) if (actual_1d is not None and pd.notna(row['predicted_move_pct_1d'])) else None
    mag_err_5d = round(abs(float(row['predicted_move_pct_5d']) - actual_5d), 4) if (actual_5d is not None and pd.notna(row['predicted_move_pct_5d'])) else None

    bt_id = row.get('existing_bt_id') or str(uuid.uuid4())

    results.append({
        'backtest_id': bt_id,
        'impact_id': row['impact_id'],
        'event_id': row['event_id'],
        'ticker': ticker,
        'headline': str(row['headline'])[:500] if row['headline'] else None,
        'direction_predicted': row['direction'],
        'magnitude_predicted': int(row['magnitude']),
        'predicted_move_1d': float(row['predicted_move_pct_1d']) if pd.notna(row['predicted_move_pct_1d']) else None,
        'predicted_move_5d': float(row['predicted_move_pct_5d']) if pd.notna(row['predicted_move_pct_5d']) else None,
        'confidence': float(row['confidence']) if pd.notna(row['confidence']) else None,
        'event_date': event_dt,
        'price_date_t0': t0_date,
        'price_date_t1': t1_date,
        'price_date_t5': t5_date,
        'actual_close_t0': t0_close,
        'actual_close_t1': t1_close,
        'actual_close_t5': t5_close,
        'actual_move_1d_pct': actual_1d,
        'actual_move_5d_pct': actual_5d,
        'direction_correct_1d': dir_correct_1d,
        'direction_correct_5d': dir_correct_5d,
        'magnitude_error_1d': mag_err_1d,
        'magnitude_error_5d': mag_err_5d,
        'scored_at': now,
    })

results_pdf = pd.DataFrame(results)
scored_1d = results_pdf['actual_move_1d_pct'].notna().sum()
scored_5d = results_pdf['actual_move_5d_pct'].notna().sum()
pending = len(results_pdf) - scored_1d
print(f"Backtest results: {len(results_pdf)} total")
print(f"  Scored (1-day): {scored_1d}")
print(f"  Scored (5-day): {scored_5d}")
print(f"  Pending (no price yet): {pending}")

## Step 4: Accuracy report

In [0]:
if scored_1d > 0:
    correct_1d = results_pdf['direction_correct_1d'].dropna()
    accuracy_1d = correct_1d.mean() * 100
    avg_err_1d = results_pdf['magnitude_error_1d'].dropna().mean()
    print(f"=== 1-DAY ACCURACY ===")
    print(f"Direction correct: {int(correct_1d.sum())}/{len(correct_1d)} ({accuracy_1d:.1f}%)")
    print(f"Avg magnitude error: {avg_err_1d:.2f}%")
    print()
    for d in ['positive', 'negative', 'neutral']:
        subset = results_pdf[results_pdf['direction_predicted'] == d]
        if len(subset) > 0 and subset['direction_correct_1d'].notna().any():
            acc = subset['direction_correct_1d'].dropna().mean() * 100
            print(f"  {d}: {acc:.0f}% accurate ({len(subset)} predictions)")
else:
    print("No 1-day results yet (prices not available). Run again after market close.")

if scored_5d > 0:
    correct_5d = results_pdf['direction_correct_5d'].dropna()
    accuracy_5d = correct_5d.mean() * 100
    print(f"\n=== 5-DAY ACCURACY ===")
    print(f"Direction correct: {int(correct_5d.sum())}/{len(correct_5d)} ({accuracy_5d:.1f}%)")

## Step 5: Write results to Delta

In [0]:
from delta.tables import DeltaTable

if len(results_pdf) == 0:
    print("No results to write.")
    dbutils.notebook.exit("no_results")

results_sdf = spark.createDataFrame(results_pdf)

tgt = DeltaTable.forName(spark, T_BACKTEST)
(
    tgt.alias("t")
    .merge(results_sdf.alias("s"), "t.impact_id = s.impact_id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

final_count = spark.sql(f"SELECT COUNT(*) as c FROM {T_BACKTEST}").collect()[0].c
scored_count = spark.sql(f"SELECT COUNT(*) as c FROM {T_BACKTEST} WHERE actual_close_t1 IS NOT NULL").collect()[0].c
print(f"Backtest table: {final_count} total rows, {scored_count} fully scored (1-day)")

## Step 6: Per-ticker scorecard

In [0]:
display(spark.sql(f"""
    SELECT ticker,
           COUNT(*) as predictions,
           SUM(CASE WHEN direction_correct_1d = true THEN 1 ELSE 0 END) as correct_1d,
           ROUND(AVG(CASE WHEN direction_correct_1d IS NOT NULL THEN CAST(direction_correct_1d AS INT) END) * 100, 1) as accuracy_pct,
           ROUND(AVG(magnitude_error_1d), 2) as avg_mag_error,
           ROUND(AVG(actual_move_1d_pct), 2) as avg_actual_move,
           SUM(CASE WHEN actual_close_t1 IS NULL THEN 1 ELSE 0 END) as pending
    FROM {T_BACKTEST}
    GROUP BY ticker
    ORDER BY predictions DESC
"""))